# Clario — LLaMA 3.2 3B Fine-Tuning v2 (QLoRA)
### Multi-label category · 3-class sentiment · chain-of-thought distillation

**Base model**: `meta-llama/Llama-3.2-3B-Instruct` (fresh from Meta — this is **not** a continuation of any previously fine-tuned Clario checkpoint)
**Method**: QLoRA, 4-bit NF4 quantization
**Data**: `train_with_cot.csv` (10,000 rows, includes Gemini's chain-of-thought `reasoning`) + `test.csv` (3,184 rows, labels only — held out, stratified to mirror the full dataset's joint label distribution)

**Tasks learned**:
- `priority`: one of `Low / Medium / High / Critical`
- `sentiment`: one of `Neutral / Negative / Frustrated` (Gemini's own sentiment-reasoning step is what promotes a ticket to Frustrated)
- `category`: **multi-label** — one or more of the 12 canonical categories below (a ticket can be e.g. both `Billing & Invoicing` and `Subscription Management`)

The model is trained to *think first* (reproduce Gemini's step-by-step reasoning) and then emit a final JSON verdict. At test time we only grade the final JSON — the reasoning text itself is never scored.

> **Kaggle setup**
> 1. Notebook → Add-ons → Secrets → add `HF_TOKEN` (a Hugging Face token with access to `meta-llama/Llama-3.2-3B-Instruct`) → toggle *Attach to notebook*.
> 2. Attach a dataset containing `train_with_cot.csv`, `test.csv`, and `label_maps.json` (from `ml_finetuning/data/splits/`).
> 3. Use a GPU accelerator (T4 x2 or P100 both work; QLoRA keeps the base model in 4-bit).


In [ ]:
# transformers/tokenizers/datasets/accelerate/peft/trl are pinned exactly:
# trl's newer loss-accounting path (chunked cross-entropy / num_valid_tokens
# bookkeeping) is incompatible with a PEFT-wrapped 4-bit model's forward on
# several recent trl releases, and crashes a few seconds into training with:
#   'CausalLMOutputWithPast' object has no attribute 'num_valid_tokens'
# This combination is verified to work for QLoRA SFT on Llama 3.2.
#
# bitsandbytes is deliberately NOT pinned to an exact version: it ships a
# compiled CUDA binary, and an old pin (e.g. 0.44.x) predates newer CUDA
# toolkits (cu121+/cu128) and falls back to a Triton-based path that breaks
# against current Triton releases (`ModuleNotFoundError: No module named
# 'triton.ops'`). Only a floor is set to rule out that old, broken range —
# pip resolves the newest release, which ships a binary matching whatever
# CUDA build the current Kaggle/Colab image's PyTorch is on.
#
# This cell installs for real when run (no leading '#' on the !pip line) --
# on a fresh Kaggle/Colab session, restart the runtime once after it
# finishes so nothing from before the install stays imported in memory.
!pip install -q \
    transformers==4.46.3 \
    tokenizers==0.20.3 \
    datasets==3.1.0 \
    accelerate==1.1.1 \
    peft==0.13.2 \
    trl==0.12.1 \
    "bitsandbytes>=0.45.0" \
    scikit-learn pandas matplotlib seaborn huggingface_hub


In [ ]:
import json
import os
import re
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, PeftModel, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")


In [ ]:
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
MAX_SEQ_LENGTH = 1024
RANDOM_STATE = 42
NUM_TRAIN_EPOCHS = 3

# ---- Data location -------------------------------------------------------
# On Kaggle, point this at the attached dataset directory, e.g.:
#   DATA_DIR_HINT = "/kaggle/input/clario-llama32-splits"
# Locally it falls back to the repo's data/splits/ directory.
DATA_DIR_HINT = "/kaggle/input/clario-llama32-splits"
LOCAL_FALLBACK = "/home/ranuga-weerasekara/Desktop/clario/ml_finetuning/data/splits"

# ---- Output location -------------------------------------------------------
# Kaggle only allows writes under /kaggle/working.
if os.path.isdir("/kaggle/working"):
    OUTPUT_DIR = "/kaggle/working/llama32_clario_v2"
else:
    OUTPUT_DIR = "/home/ranuga-weerasekara/Desktop/clario/ml_finetuning/models/llama32_clario_v2"

CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
FINAL_DIR = os.path.join(OUTPUT_DIR, "final")
REPORT_DIR = os.path.join(OUTPUT_DIR, "reports")
for d in (OUTPUT_DIR, CHECKPOINT_DIR, FINAL_DIR, REPORT_DIR):
    os.makedirs(d, exist_ok=True)

print(f"OUTPUT_DIR: {OUTPUT_DIR}")


In [ ]:
def resolve_data_dir(hint: str, fallback: str) -> str:
    """Find the directory containing train_with_cot.csv, searching the Kaggle
    input mount first (it may be nested one level deeper than the dataset
    slug), then falling back to a local path."""
    for base in (hint, "/kaggle/input", fallback):
        p = Path(base)
        if not p.exists():
            continue
        matches = sorted(p.rglob("train_with_cot.csv"))
        if matches:
            return str(matches[0].parent)
    raise FileNotFoundError(
        "Could not find train_with_cot.csv under any of: "
        f"{hint}, /kaggle/input, {fallback}"
    )


DATA_DIR = resolve_data_dir(DATA_DIR_HINT, LOCAL_FALLBACK)
print(f"[Data] Using data directory: {DATA_DIR}")


In [ ]:
train_full_df = pd.read_csv(os.path.join(DATA_DIR, "train_with_cot.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

with open(os.path.join(DATA_DIR, "label_maps.json")) as f:
    LABEL_MAPS = json.load(f)

CATEGORY_LABELS = LABEL_MAPS["categories"]
PRIORITY_LABELS = LABEL_MAPS["priorities"]
SENTIMENT_LABELS = LABEL_MAPS["sentiments"]

# category column is stored as a JSON array string, e.g. '["Billing & Invoicing", "Refunds"]'
train_full_df["category_list"] = train_full_df["category"].apply(json.loads)
test_df["category_list"] = test_df["category"].apply(json.loads)

print(f"Train pool: {len(train_full_df)} | Held-out test: {len(test_df)}")
print(f"Categories ({len(CATEGORY_LABELS)}): {CATEGORY_LABELS}")
print(f"Priorities: {PRIORITY_LABELS}")
print(f"Sentiments: {SENTIMENT_LABELS}")


## Carve a validation slice out of the training pool

The held-out `test.csv` (3,184 rows) stays untouched until final evaluation. For early-stopping / checkpoint selection during training we need a small validation slice, so we split 10% off the 10,000-row training pool (stratified by `priority`, the field with the most extreme class imbalance — `Critical` is only ~1.6% of the data).

In [ ]:
df_train, df_val = train_test_split(
    train_full_df,
    test_size=0.10,
    random_state=RANDOM_STATE,
    stratify=train_full_df["priority"],
)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test (held out): {len(test_df)}")


In [ ]:
SYSTEM_PROMPT = (
    "You are Clario, an intelligent IT support ticket triage assistant.\n"
    "Given a product name and issue description, think step by step covering: "
    "user intent, urgency, tone/sentiment, and category. Then output your final answer.\n\n"
    "Respond with your step-by-step reasoning first, then on a new line output ONLY "
    "the final JSON in this exact format (category is a LIST — a ticket may belong "
    "to more than one category):\n"
    '{"priority": "<Low|Medium|High|Critical>", '
    '"sentiment": "<Neutral|Negative|Frustrated>", '
    '"category": ["<one or more of: ' + ", ".join(CATEGORY_LABELS) + '>"]}'
)

print(SYSTEM_PROMPT)


In [ ]:
def build_user_turn(product: str, issue: str) -> str:
    return f"Product: {product}\nIssue: {issue}"


def build_train_text(row) -> str:
    """Full chat-formatted example: system + user + assistant(reasoning -> JSON).
    The model is trained to reproduce the reasoning AND the final JSON."""
    final_json = json.dumps({
        "priority": row["priority"],
        "sentiment": row["sentiment"],
        "category": row["category_list"],
    })
    assistant_msg = f"{row['reasoning']}\n{final_json}"
    return (
        "<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n{SYSTEM_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n{build_user_turn(row['input_product'], row['input_issue_description'])}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n{assistant_msg}<|eot_id|>"
    )


def build_inference_prompt(product: str, issue: str) -> str:
    """System + user only, left open at the assistant header for generation."""
    return (
        "<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n{SYSTEM_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n{build_user_turn(product, issue)}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n"
    )


df_train["text"] = df_train.apply(build_train_text, axis=1)
df_val["text"] = df_val.apply(build_train_text, axis=1)

ds_train = Dataset.from_pandas(df_train[["text"]].reset_index(drop=True))
ds_val = Dataset.from_pandas(df_val[["text"]].reset_index(drop=True))

print(ds_train[0]["text"][:800])


In [ ]:
# ---- Authenticate with Hugging Face -----------------------------------------
# On Kaggle: Notebook -> Add-ons -> Secrets -> add HF_TOKEN -> Attach to notebook.
# Locally:   export HF_TOKEN=hf_xxx...
from huggingface_hub import login

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("[Auth] Loaded HF_TOKEN from Kaggle Secrets")
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if HF_TOKEN:
        print("[Auth] Loaded HF_TOKEN from environment variable")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN not found. On Kaggle, add it under Notebook -> Add-ons -> Secrets "
        "and toggle 'Attach to notebook'. Locally, `export HF_TOKEN=hf_xxx...`."
    )

login(token=HF_TOKEN)


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    token=HF_TOKEN,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)


In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## Checkpointing

Every `save_steps` interval writes a full checkpoint (LoRA adapter weights, optimizer/scheduler state, RNG state) to `CHECKPOINT_DIR`. `save_total_limit` is left unset so **no checkpoint is ever pruned** — since QLoRA only checkpoints the small adapter (a few tens of MB, not the 4-bit base model), keeping every one of them is cheap. This also means a Kaggle session timeout doesn't cost you the run: re-executing this notebook auto-resumes from the latest checkpoint (see the training cell below).

In [ ]:
training_args = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=None,          # keep every checkpoint
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True,
    max_grad_norm=0.3,
    report_to="none",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
    seed=RANDOM_STATE,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    args=training_args,
    tokenizer=tokenizer,
)


In [ ]:
# Resume automatically if this cell has already produced checkpoints
# (handles Kaggle session interruptions without losing progress).
last_checkpoint = None
if os.path.isdir(CHECKPOINT_DIR):
    existing = [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith("checkpoint-")]
    if existing:
        last_checkpoint = os.path.join(
            CHECKPOINT_DIR,
            sorted(existing, key=lambda d: int(d.split("-")[1]))[-1],
        )
        print(f"[Resume] Found existing checkpoint, resuming from: {last_checkpoint}")

training_succeeded = False
train_result = None
try:
    print("Starting fine-tuning...")
    train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
    training_succeeded = True
    print("Training complete.")
except Exception as exc:
    print(f"[WARNING] Training raised an exception: {exc}")
    print("The most recent step checkpoint under CHECKPOINT_DIR is still on disk. "
          "Saving the model's current in-memory state below regardless, then re-run "
          "this cell to resume from the last checkpoint.")


## Save the model

This cell runs immediately after training and always leaves a usable model on disk, whether training finished all `NUM_TRAIN_EPOCHS` cleanly or was interrupted:
- if training **succeeded**, `trainer.model` already holds the *best* checkpoint by `eval_loss` (`load_best_model_at_end=True`), so `FINAL_DIR` gets the best-performing adapter, not just the last one;
- if training **raised partway through**, this still saves whatever state `trainer.model` is currently holding, so the run isn't a total loss — the step checkpoints in `CHECKPOINT_DIR` remain available to resume from either way.

Everything needed to reload the model later (adapter weights, tokenizer, label maps, a training summary) is written to `FINAL_DIR`, and the whole run (all checkpoints + final model + reports) is additionally zipped into one Kaggle output artifact.

In [ ]:
os.makedirs(FINAL_DIR, exist_ok=True)

trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

with open(os.path.join(FINAL_DIR, "label_maps.json"), "w") as f:
    json.dump(LABEL_MAPS, f, indent=2)

training_summary = {
    "base_model": MODEL_ID,
    "training_succeeded": training_succeeded,
    "resumed_from_checkpoint": last_checkpoint,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "best_checkpoint_step": getattr(trainer.state, "best_global_step", None),
    "best_eval_loss": getattr(trainer.state, "best_metric", None),
    "train_runtime_sec": (train_result.metrics.get("train_runtime") if train_result else None),
}
with open(os.path.join(FINAL_DIR, "training_summary.json"), "w") as f:
    json.dump(training_summary, f, indent=2)

print(f"[Saved] Final adapter + tokenizer + metadata -> {FINAL_DIR}")
print(f"[Saved] All step checkpoints retained -> {CHECKPOINT_DIR}")
print(json.dumps(training_summary, indent=2))


In [ ]:
# Bundle the full run (checkpoints + final model + reports, once they exist)
# into a single zip so it survives as one downloadable Kaggle output artifact.
zip_base = os.path.join(os.path.dirname(OUTPUT_DIR), os.path.basename(OUTPUT_DIR) + "_run")
zip_path = shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
print(f"[Saved] Full run zipped -> {zip_path}")


## Evaluation

Reload the saved adapter fresh from `FINAL_DIR` (rather than reusing the in-memory `trainer.model`) so evaluation exercises exactly what got persisted to disk. The held-out `test.csv` (3,184 rows) is graded on **priority, sentiment, and category only** — the model is still prompted to reason first (since that's how it was trained), but the reasoning text is discarded; only the trailing JSON block is parsed and scored.

In [ ]:
del model
torch.cuda.empty_cache()

eval_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    token=HF_TOKEN,
)
eval_model = PeftModel.from_pretrained(eval_base, FINAL_DIR)
eval_model.eval()


In [ ]:
def extract_final_json(generated_text: str) -> dict:
    """Pull the trailing {...} JSON block out of the model's reasoning + answer
    output. Falls back to empty/Unknown fields if nothing parses."""
    start = generated_text.rfind("{")
    end = generated_text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return {"priority": "Unknown", "sentiment": "Unknown", "category": []}
    try:
        parsed = json.loads(generated_text[start:end + 1])
    except json.JSONDecodeError:
        return {"priority": "Unknown", "sentiment": "Unknown", "category": []}
    category = parsed.get("category", [])
    if isinstance(category, str):
        category = [category]
    return {
        "priority": parsed.get("priority", "Unknown"),
        "sentiment": parsed.get("sentiment", "Unknown"),
        "category": category,
    }


def predict_single(product: str, issue: str, model, tokenizer, device="cuda") -> dict:
    prompt = build_inference_prompt(product, issue)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return extract_final_json(generated)


In [ ]:
from tqdm.auto import tqdm

y_true_priority, y_pred_priority = [], []
y_true_sentiment, y_pred_sentiment = [], []
y_true_category, y_pred_category = [], []  # lists of label-sets

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Evaluating"):
    pred = predict_single(row["input_product"], row["input_issue_description"], eval_model, tokenizer)

    y_true_priority.append(row["priority"])
    y_pred_priority.append(pred["priority"])

    y_true_sentiment.append(row["sentiment"])
    y_pred_sentiment.append(pred["sentiment"])

    y_true_category.append(row["category_list"])
    y_pred_category.append(pred["category"])

print(f"Evaluated {len(test_df)} held-out test tickets.")


## Reporting

Each task gets a per-class Precision / Recall / F1 table plus Overall Accuracy, Macro Avg, and Weighted Avg — rendered both as a printed table and as a dark-themed image, matching the reporting style used for the other Clario model evaluations.

In [ ]:
def render_report_table(rows_df: pd.DataFrame, overall_accuracy: float, title: str, save_path: str):
    """Render a classification-report DataFrame as a dark-themed table image."""
    n_rows = len(rows_df) + 3  # + Overall Accuracy / Macro Avg / Weighted Avg spacer rows
    fig, ax = plt.subplots(figsize=(9, 0.5 * n_rows + 2))
    fig.patch.set_facecolor("#111318")
    ax.set_facecolor("#111318")
    ax.axis("off")

    display_df = rows_df.copy()
    for col in ["precision", "recall", "f1-score"]:
        display_df[col] = display_df[col].map(lambda v: f"{v:.2f}")

    cell_text = display_df[["precision", "recall", "f1-score"]].values.tolist()
    cell_text.append([f"{overall_accuracy:.3f}", "", ""])
    row_labels = list(display_df.index) + ["Overall Accuracy"]

    table = ax.table(
        cellText=cell_text,
        rowLabels=row_labels,
        colLabels=["Precision", "Recall", "F1-Score"],
        cellLoc="center",
        rowLoc="left",
        loc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1, 1.8)

    n_data_rows = len(cell_text)
    summary_start = len(display_df)  # index (0-based, after header row offset) of the Overall Accuracy row
    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor("#333844")
        cell.set_text_props(color="white")
        if row == 0:
            cell.set_facecolor("#0d0f13")
            cell.set_text_props(color="#7ee7ff", weight="bold")
        elif row == summary_start + 1:  # +1 because table rows are 1-indexed past the header
            cell.set_facecolor("#1f2430")
            cell.set_text_props(color="#ffd479", weight="bold")
        else:
            cell.set_facecolor("#1b1f27" if row % 2 else "#15181e")

    ax.set_title(title, color="white", fontsize=14, fontweight="bold", pad=20)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, facecolor=fig.get_facecolor())
    plt.show()


In [ ]:
# ---- Priority (single-label, 4 classes) -------------------------------------
priority_report = classification_report(
    y_true_priority, y_pred_priority, labels=PRIORITY_LABELS,
    output_dict=True, zero_division=0,
)
priority_df = pd.DataFrame(priority_report).transpose().loc[PRIORITY_LABELS]
priority_acc = accuracy_score(y_true_priority, y_pred_priority)

print("=" * 60)
print("PRIORITY CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_true_priority, y_pred_priority, labels=PRIORITY_LABELS, zero_division=0))
print(f"Macro Avg   : {priority_report['macro avg']}")
print(f"Weighted Avg: {priority_report['weighted avg']}")

render_report_table(
    priority_df, priority_acc, "Priority — Fine-Tuned Llama 3.2 3B",
    os.path.join(REPORT_DIR, "priority_report.png"),
)


In [ ]:
# ---- Sentiment (single-label, 3 classes) ------------------------------------
sentiment_report = classification_report(
    y_true_sentiment, y_pred_sentiment, labels=SENTIMENT_LABELS,
    output_dict=True, zero_division=0,
)
sentiment_df = pd.DataFrame(sentiment_report).transpose().loc[SENTIMENT_LABELS]
sentiment_acc = accuracy_score(y_true_sentiment, y_pred_sentiment)

print("=" * 60)
print("SENTIMENT CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_true_sentiment, y_pred_sentiment, labels=SENTIMENT_LABELS, zero_division=0))

render_report_table(
    sentiment_df, sentiment_acc, "Sentiment — Fine-Tuned Llama 3.2 3B",
    os.path.join(REPORT_DIR, "sentiment_report.png"),
)


In [ ]:
# ---- Category (multi-label, 12 classes) -------------------------------------
def to_indicator_matrix(label_sets, labels):
    return np.array([[1 if lbl in s else 0 for lbl in labels] for s in label_sets])

Y_true_cat = to_indicator_matrix(y_true_category, CATEGORY_LABELS)
Y_pred_cat = to_indicator_matrix(y_pred_category, CATEGORY_LABELS)

category_report = classification_report(
    Y_true_cat, Y_pred_cat, target_names=CATEGORY_LABELS,
    output_dict=True, zero_division=0,
)
category_df = pd.DataFrame(category_report).transpose().loc[CATEGORY_LABELS]

# Per-label accuracy averaged across all 12 binary category flags (headline metric).
per_label_accuracy = np.mean([
    accuracy_score(Y_true_cat[:, i], Y_pred_cat[:, i]) for i in range(len(CATEGORY_LABELS))
])
# Exact-set-match accuracy: the whole predicted category set matches exactly (secondary, stricter metric).
exact_match_accuracy = np.mean(np.all(Y_true_cat == Y_pred_cat, axis=1))

print("=" * 60)
print("CATEGORY CLASSIFICATION REPORT (multi-label)")
print("=" * 60)
print(classification_report(Y_true_cat, Y_pred_cat, target_names=CATEGORY_LABELS, zero_division=0))
print(f"Per-label Accuracy (headline) : {per_label_accuracy:.4f}")
print(f"Exact-Set-Match Accuracy       : {exact_match_accuracy:.4f}")

render_report_table(
    category_df, per_label_accuracy, "Category (multi-label) — Fine-Tuned Llama 3.2 3B",
    os.path.join(REPORT_DIR, "category_report.png"),
)


In [ ]:
summary = pd.DataFrame({
    "Task": ["Priority", "Sentiment", "Category (per-label)", "Category (exact-set match)"],
    "Accuracy": [priority_acc, sentiment_acc, per_label_accuracy, exact_match_accuracy],
})
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(summary.to_string(index=False))
summary.to_csv(os.path.join(REPORT_DIR, "summary.csv"), index=False)


In [ ]:
results_df = test_df[["input_product", "input_issue_description", "priority", "sentiment"]].copy()
results_df["category_true"] = [json.dumps(s) for s in y_true_category]
results_df["pred_priority"] = y_pred_priority
results_df["pred_sentiment"] = y_pred_sentiment
results_df["pred_category"] = [json.dumps(s) for s in y_pred_category]
results_df["priority_correct"] = results_df["priority"] == results_df["pred_priority"]
results_df["sentiment_correct"] = results_df["sentiment"] == results_df["pred_sentiment"]
results_df["category_exact_match"] = [set(t) == set(p) for t, p in zip(y_true_category, y_pred_category)]

results_csv = os.path.join(REPORT_DIR, "test_results.csv")
results_df.to_csv(results_csv, index=False)
print(f"Detailed per-ticket results saved to: {results_csv}")


In [ ]:
demo_cases = [
    ("Video Classroom", "The video player keeps buffering and the course is unusable. I want a refund."),
    ("Payment & Billing", "I was charged twice this month for the same subscription."),
    ("Student Web Portal", "Can you add a dark mode to the portal?"),
    ("Assessment Module", "Cannot login to complete my final assessment exam."),
]

print("=" * 60)
print("DEMO INFERENCE")
print("=" * 60)
for product, issue in demo_cases:
    pred = predict_single(product, issue, eval_model, tokenizer)
    print(f"\nProduct : {product}")
    print(f"Issue   : {issue}")
    print(f"▶ Priority={pred['priority']}  Sentiment={pred['sentiment']}  Category={pred['category']}")


## Where everything landed

- **Every step checkpoint** (adapter weights + optimizer/scheduler/RNG state): `CHECKPOINT_DIR` — nothing pruned, so a Kaggle session timeout only costs you a re-run, not lost progress (the training cell auto-resumes from the latest one).
- **Final model** (best checkpoint by `eval_loss`, adapter + tokenizer + label maps + training summary): `FINAL_DIR`.
- **Evaluation report images + CSVs**: `REPORT_DIR`.
- **Everything zipped together** as one Kaggle output artifact: `<OUTPUT_DIR>_run.zip`.

All four are printed with their absolute paths above the corresponding cells if you need to re-locate them.